In [1165]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns

Extractions des dataframes clubs et matchs from 2013 to 2024. Le but étant d'enrichir le tableau des anciens matchs grâces aux infos des clubs

In [1166]:
clubs = pd.read_csv("data/clubs_fr.csv")

matchs_2013_2024 = pd.read_csv("data/matchs_2013_2024.csv", index_col=0)

# print("Clubs :")
# display(clubs.head())
# print()

# print("Matchs from 2013 to 2024 :")
# display(matchs_2013_2024.head())

In [1167]:
clubs = clubs.drop(columns=["foreigners_number", "foreigners_percentage", "club_code", "net_transfer_record", "stadium_name", "domestic_competition_id", "name", "coach_name"])
# print("Clubs clean :")
# display(clubs.head())
# print()

matchs_2013_2024 = matchs_2013_2024.drop(columns=["home_club_formation", "away_club_formation", "home_club_goals", "away_club_goals", "aggregate", "competition_type"])
matchs_2013_2024 = matchs_2013_2024.dropna(subset="results")
# print("Matchs 2013 2024 clean :")
# display(matchs_2013_2024.head())

In [1168]:
matchs_2013_2024 = pd.merge(matchs_2013_2024, clubs, left_on="home_club_id", right_on="club_id")
matchs_2013_2024 = matchs_2013_2024.drop(columns=["club_id"])
matchs_2013_2024 = matchs_2013_2024.rename(columns={"squad_size": "home_club_squad_size", "average_age": "home_club_average_age", "national_team_players": "home_club_national_team_players"})
# print(matchs_2013_2024.columns)
# display(matchs_2013_2024.head())

In [1169]:
matchs_2013_2024 = pd.merge(matchs_2013_2024, clubs, left_on="away_club_id", right_on="club_id")
matchs_2013_2024 = matchs_2013_2024.drop(columns=["club_id", "stadium_seats_y"])
matchs_2013_2024 = matchs_2013_2024.rename(columns={"squad_size": "away_club_squad_size", "average_age": "away_club_average_age", "national_team_players": "away_club_national_team_players", "stadium_seats_x": "stadium_seats"})
print("All matchs from 2013 to 2024 :")
display(matchs_2013_2024.head())

All matchs from 2013 to 2024 :


,game_id,season,round,date,home_club_id,away_club_id,home_club_position,away_club_position,home_club_manager_name,away_club_manager_name,...,home_club_name,away_club_name,results,home_club_squad_size,home_club_average_age,home_club_national_team_players,stadium_seats,away_club_squad_size,away_club_average_age,away_club_national_team_players
0,2223841,2012,6. Matchday,2012-09-22,3911,1423,8.0,9.0,Landry Chauvin,Daniel Sanchez,...,Stade brestois 29,Valenciennes FC,1,23,25.3,6,15220,30,23.6,5
1,2223842,2012,7. Matchday,2012-09-29,1147,3911,9.0,10.0,Alex Dupont,Landry Chauvin,...,AC Ajaccio,Stade brestois 29,1,24,27.5,6,10446,23,25.3,6
2,2223843,2012,6. Matchday,2012-09-22,1421,1159,5.0,19.0,Hubert Fournier,Jean Fernandez,...,Stade Reims,AS Nancy-Lorraine,1,24,23.9,7,20519,27,25.7,0
3,2223844,2012,6. Matchday,2012-09-21,969,618,16.0,10.0,René Girard,Christophe Galtier,...,Montpellier HSC,AS Saint-Étienne,0,24,24.2,3,22000,29,24.0,6
4,2223845,2012,6. Matchday,2012-09-23,1082,1041,12.0,2.0,Rudi Garcia,Rémi Garde,...,Lille Olympique Sporting Club,Olympique Lyonnais,0,27,26.5,11,50186,28,25.5,11


=============================================================================================================================================================================================

Extraction du dataframe player_valuation_before_season. L'objectif est de calculer, pour chaque année et donc pour chaque match, quelle sont les valeurs marchandes des équipes. Car une équipe ayant une valeur marchande plus élevée que son adverse a une probabilité plus élevée de remporter le match

In [1170]:
player_valuation_before_season = pd.read_csv("data/player_valuation_before_season.csv")
# display(player_valuation_before_season.head())

In [1171]:
player_valuation_before_season["date"] = pd.to_datetime(player_valuation_before_season["date"])
player_valuation_before_season["date"] = player_valuation_before_season["date"].dt.year
player_valuation_before_season = player_valuation_before_season.rename(columns={"date": "year"})
# display(player_valuation_before_season.head())
club_valueation_per_year = player_valuation_before_season.groupby(["current_club_id", "year"])["market_value_in_eur"].sum().reset_index()
# display(club_valueation_per_year.head())

matchs_2013_2024 = pd.merge(matchs_2013_2024, club_valueation_per_year, left_on=["home_club_id", "season"], right_on=["current_club_id", "year"])
matchs_2013_2024 = matchs_2013_2024.drop(columns=["current_club_id", "year"])
# print(matchs_2013_2024.columns)
matchs_2013_2024 = matchs_2013_2024.rename(columns={"market_value_in_eur": "home_club_value_in_eur"})
# display(matchs_2013_2024.head())


matchs_2013_2024 = pd.merge(matchs_2013_2024, club_valueation_per_year, left_on=["away_club_id", "season"], right_on=["current_club_id", "year"])
matchs_2013_2024 = matchs_2013_2024.drop(columns=["current_club_id", "year"])
# print(matchs_2013_2024.columns)
matchs_2013_2024 = matchs_2013_2024.rename(columns={"market_value_in_eur": "away_club_value_in_eur"})
print("Matchs de 2013 à 2024 avec les valeurs marchandes des équipes :")
display(matchs_2013_2024.head())

Matchs de 2013 à 2024 avec les valeurs marchandes des équipes :


,game_id,season,round,date,home_club_id,away_club_id,home_club_position,away_club_position,home_club_manager_name,away_club_manager_name,...,results,home_club_squad_size,home_club_average_age,home_club_national_team_players,stadium_seats,away_club_squad_size,away_club_average_age,away_club_national_team_players,home_club_value_in_eur,away_club_value_in_eur
0,2223841,2012,6. Matchday,2012-09-22,3911,1423,8.0,9.0,Landry Chauvin,Daniel Sanchez,...,1,23,25.3,6,15220,30,23.6,5,64150000,29550000
1,2223842,2012,7. Matchday,2012-09-29,1147,3911,9.0,10.0,Alex Dupont,Landry Chauvin,...,1,24,27.5,6,10446,23,25.3,6,61575000,64150000
2,2223843,2012,6. Matchday,2012-09-22,1421,1159,5.0,19.0,Hubert Fournier,Jean Fernandez,...,1,24,23.9,7,20519,27,25.7,0,88700000,61450000
3,2223844,2012,6. Matchday,2012-09-21,969,618,16.0,10.0,René Girard,Christophe Galtier,...,0,24,24.2,3,22000,29,24.0,6,164525000,177000000
4,2223845,2012,6. Matchday,2012-09-23,1082,1041,12.0,2.0,Rudi Garcia,Rémi Garde,...,0,27,26.5,11,50186,28,25.5,11,101050000,145700000
